# 🛡️ PEDAS 2026 - Data Preprocessing & Feature Engineering
**Platform Evaluasi Data Sains (PEDAS) 2026**
*Pipeline: Data Loading ➔ Label Normalization ➔ Mislabel Audit & Relabeling ➔ Feature Engineering*
---


## 1. Setup Lingkungan & Import Library
Tahap inisialisasi modul yang digunakan:
- `pandas` untuk manipulasi data tabular.
- `re` untuk penanganan pola regular expression (validasi format IP, pemindaian kata kunci).
- `os` untuk pengelolaan direktori penyimpanan output.


In [3]:
import sys
import os
import re
import pandas as pd
sys.path.insert(0, '../')

# Pengaturan opsi tampilan pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)


## 2. Setup & Pemuatan Data
Memuat dataset mentah dari folder `../../data/raw/`:
- `training.csv`: Data latih berisi fitur URL, IP, domain, dan label kategori.
- `predict.csv`: Data uji untuk evaluasi prediksi.


In [4]:
# Load dataset training dan predict
train = pd.read_csv('../../data/raw/training.csv')
test = pd.read_csv('../../data/raw/predict.csv')

print(f"Dimensi Training : {train.shape}")
print(f"Dimensi Predict  : {test.shape}")
train.head(3)


Dimensi Training : (8400, 10)
Dimensi Predict  : (1500, 10)


,url,brand,discovered,confidence_level,ip,domain,sld,category,registrar,registration_date
0,https://lucah3.***********.my.id/,Telegram,5/1/2024 21:52,100,NaN,***********.my.id,my.id,phishingg,PT Web Media Technology Indonesia,5/14/2026
1,https://dpmptsp.*********.go.id/petapotensi/da...,judi online,8/5/2024 20:41,100,103.162.68.84,*********.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,5/11/2009
2,http://klikpad.bkpd.**************.go.id/klikp...,-,6/14/2024 7:05,100,103.18.117.8,**************.go.id,go.id,online gambling,Kementerian Komunikasi dan Informatika,3/13/2008


## 3. Pembersihan & Standardisasi Label Kategori
Melakukan normalisasi teks kategori target (`category`) pada data latih:
1. Menghilangkan spasi awal/akhir (*strip*).
2. Memperbaiki *typo* umum (seperti `online gamblingg`, `phishingg`, `spamm`, dsb.).
3. Menyeragamkan teks menjadi huruf kecil (*lowercase*).


In [5]:
# Mapping typo dan variasi penulisan kategori
cat_map = {
    'online gamblingg': 'online gambling',
    'Online Gambling': 'online gambling',
    'phishingg': 'phishing',
    'otherr': 'other',
    'Other': 'other',
    'malwaree': 'malware',
    'spamm': 'spam',
    'Brand': 'brand',
    'FakeShop': 'fakeshop',
    'PIIExposure': 'pii_exposure',
}

# Terapkan pembersihan pada kolom category
train['category'] = train['category'].str.strip().replace(cat_map).str.lower().str.strip()

print("=== Distribusi Kategori Training ===")
print(train['category'].value_counts())


=== Distribusi Kategori Training ===
category
online gambling    5447
phishing           2253
other               284
spam                185
malware             179
brand                45
fakeshop              5
violence              1
pii_exposure          1
Name: count, dtype: int64


## 4. Audit Mislabeling & Relabeling Otomatis
Pemeriksaan anomali label (*mislabeling*) pada training set dengan alur korelasi bertingkat:
`Category` ➔ `URL Keyword` ➔ `IP Exact` ➔ `Subnet /24` ➔ `Domain`.

**Aturan Penilaian Kecurigaan (`suspect_score`):**
- Keyword URL sesuai kategori lain: `+3` poin
- IP exact didominasi kategori lain ($\ge 80\%$ purity, min. 5 count): `+2` poin
- Subnet `/24` didominasi kategori lain ($\ge 80\%$ purity, min. 10 count): `+2` poin
- Domain didominasi kategori lain ($\ge 80\%$ purity, min. 5 count): `+1` poin

> **Catatan Penting:** Audit mislabel hanya dijalankan pada dataset `train` untuk menghindari kebocoran data (*data leakage*).


In [6]:
# ============================================================
# LANGKAH 0: Validasi format IP (WAJIB sebelum dipakai korelasi)
# ============================================================
def is_valid_ipv4(ip):
    """Cek apakah string IP valid formatnya. Return None kalau missing."""
    if pd.isna(ip):
        return None
    m = re.match(r'^(\d{1,3})\.(\d{1,3})\.(\d{1,3})\.(\d{1,3})$', str(ip).strip())
    if not m:
        return False
    return all(0 <= int(x) <= 255 for x in m.groups())


# ============================================================
# Keyword khas per kategori
# ============================================================
KEYWORD_MAP = {
    'online gambling': ['slot', 'gacor', 'judi', 'togel', 'toto', 'poker', 'rtp', 'depo'],
    'phishing': ['verify', 'login', 'account', 'secure-', 'confirm-account'],
}

# Ambang batas keandalan statistik dominasi
MIN_COUNT_IP = 5
MIN_COUNT_SUBNET = 10
MIN_COUNT_DOMAIN = 5
MIN_PURITY = 0.8

# Bobot skor kecurigaan per jenis sinyal
SCORE_KEYWORD = 3
SCORE_IP_EXACT = 2
SCORE_SUBNET = 2
SCORE_DOMAIN = 1


def _build_dominance_stats(df, groupcol, category_col='category'):
    """Hitung kategori dominan + purity + jumlah baris per grup (IP/subnet/domain)."""
    stats = df.dropna(subset=[groupcol]).groupby(groupcol)[category_col].agg(
        dominant=lambda x: x.value_counts().idxmax(),
        purity=lambda x: x.value_counts().max() / len(x),
        count='count'
    )
    return stats


def audit_mislabel(df, category_col='category', url_col='url', ip_col='ip', domain_col='domain'):
    """
    Audit kemungkinan salah label, alur: category -> url -> ip.

    PENTING: fungsi ini HANYA boleh dipakai di TRAINING SET
    (butuh label 'category' yang sudah ada untuk cross-check).
    Statistik yang dihitung di sini juga HANYA boleh dihitung dari train
    (jangan gabung dengan test/predict.csv untuk hindari data leakage).

    Return: dataframe asli + kolom tambahan:
        - suspect_score   : skor kecurigaan (semakin tinggi semakin yakin salah label)
        - suspect_reasons : daftar alasan kecurigaan (list of string)
    """
    df = df.copy()

    # ---------- LANGKAH 0: validasi format IP ----------
    df['ip_valid'] = df[ip_col].apply(is_valid_ipv4)
    df.loc[df['ip_valid'] == False, ip_col] = None  # treat invalid sebagai missing

    # ---------- LANGKAH 1: kelompokkan per category (baseline pembanding) ----------

    # ---------- LANGKAH 2: cek url (keyword per kategori) ----------
    for cat, kws in KEYWORD_MAP.items():
        col = f'kw_{cat.replace(" ", "_")}'
        df[col] = df[url_col].str.lower().str.contains('|'.join(kws), na=False)

    # ---------- LANGKAH 3: cek ip (exact + subnet) + domain ----------
    df['ip_prefix24'] = df[ip_col].str.rsplit('.', n=1).str[0]

    ip_stats = _build_dominance_stats(df, ip_col, category_col).add_prefix('ip_')
    subnet_stats = _build_dominance_stats(df, 'ip_prefix24', category_col).add_prefix('subnet_')
    domain_stats = _build_dominance_stats(df, domain_col, category_col).add_prefix('domain_')

    df = df.merge(ip_stats, left_on=ip_col, right_index=True, how='left')
    df = df.merge(subnet_stats, left_on='ip_prefix24', right_index=True, how='left')
    df = df.merge(domain_stats, left_on=domain_col, right_index=True, how='left')

    # ---------- Gabungkan jadi skor kecurigaan ----------
    def compute_score(row):
        score = 0
        reasons = []

        for cat in KEYWORD_MAP:
            col = f'kw_{cat.replace(" ", "_")}'
            if row[col] and row[category_col] != cat:
                score += SCORE_KEYWORD
                reasons.append(f"keyword_url->{cat}(+{SCORE_KEYWORD})")

        if (pd.notna(row['ip_dominant']) and row['ip_count'] >= MIN_COUNT_IP
                and row['ip_purity'] >= MIN_PURITY and row[category_col] != row['ip_dominant']):
            score += SCORE_IP_EXACT
            reasons.append(f"ip_exact->{row['ip_dominant']}(+{SCORE_IP_EXACT})")

        if (pd.notna(row['subnet_dominant']) and row['subnet_count'] >= MIN_COUNT_SUBNET
                and row['subnet_purity'] >= MIN_PURITY and row[category_col] != row['subnet_dominant']):
            score += SCORE_SUBNET
            reasons.append(f"subnet->{row['subnet_dominant']}(+{SCORE_SUBNET})")

        if (pd.notna(row['domain_dominant']) and row['domain_count'] >= MIN_COUNT_DOMAIN
                and row['domain_purity'] >= MIN_PURITY and row[category_col] != row['domain_dominant']):
            score += SCORE_DOMAIN
            reasons.append(f"domain->{row['domain_dominant']}(+{SCORE_DOMAIN})")

        return pd.Series([score, reasons])

    df[['suspect_score', 'suspect_reasons']] = df.apply(compute_score, axis=1)

    return df


def summarize_audit(df_audited, score_col='suspect_score'):
    """Cetak ringkasan hasil audit ke terminal."""
    print("=== Distribusi skor kecurigaan ===")
    print(df_audited[score_col].value_counts().sort_index(ascending=False))
    print(f"\nSkor >=5 (relabel otomatis)      : {(df_audited[score_col] >= 5).sum()} baris")
    print(f"Skor 3-4 (verifikasi manual)     : {((df_audited[score_col] >= 3) & (df_audited[score_col] < 5)).sum()} baris")
    print(f"Skor 1-2 (biarkan, terlalu lemah): {((df_audited[score_col] >= 1) & (df_audited[score_col] < 3)).sum()} baris")


def _pick_relabel_target(row):
    """
    Tentukan kategori pengganti dari suspect_reasons, prioritas sesuai bobot skor:
    keyword_url > ip_exact > subnet > domain.
    """
    reasons = row['suspect_reasons']
    if not reasons:
        return row['category']

    targets = [r.split('->')[1].split('(')[0] for r in reasons if '->' in r]
    if not targets:
        return row['category']

    return pd.Series(targets).value_counts().idxmax()


def apply_relabel(df_audited, score_threshold=5, category_col='category'):
    """
    Terapkan relabel untuk baris dengan suspect_score >= score_threshold.

    Return: (df_relabeled, diff_table)
        - df_relabeled : dataframe dengan category sudah diperbarui
        - diff_table   : tabel ringkas berisi baris yang BENERAN berubah
    """
    df = df_audited.copy()
    df['category_before'] = df[category_col]

    mask = df['suspect_score'] >= score_threshold
    df.loc[mask, category_col] = df.loc[mask].apply(_pick_relabel_target, axis=1)

    changed_mask = df[category_col] != df['category_before']
    diff_table = df.loc[changed_mask, [
        'url', 'category_before', category_col, 'suspect_score', 'suspect_reasons'
    ]].rename(columns={category_col: 'category_after'})

    return df, diff_table


def summarize_diff(diff_table):
    """Cetak ringkasan perubahan label ke terminal."""
    print(f"=== Total baris yang benar-benar berubah label: {len(diff_table)} ===\n")
    if len(diff_table) == 0:
        print("Tidak ada perubahan.")
        return
    print("=== Perubahan per pasangan (label lama -> label baru) ===")
    transition = diff_table.groupby(['category_before', 'category_after']).size().sort_values(ascending=False)
    print(transition)


### 4.1 Eksekusi Audit & Penyimpanan Relabel Data
1. Menjalankan fungsi `audit_mislabel` pada data `train`.
2. Menerapkan relabeling otomatis untuk sampel dengan `suspect_score >= 5`.
3. Menyimpan hasil audit dan data relabeled ke folder `../../data/result/v1/`.
4. Memperbarui dataframe `train` dengan data yang telah direlabel.


In [7]:
# Jalankan audit mislabel
train_audited = audit_mislabel(train)
summarize_audit(train_audited)

# Buat folder result jika belum ada
os.makedirs('../../data/result/v1', exist_ok=True)
train_audited.to_csv('../../data/result/v1/train_audit.csv', index=False)
print("\nHasil audit disimpan di data/result/v1/train_audit.csv")

# Terapkan relabel otomatis (threshold skor >= 5)
train_relabeled, diff_table = apply_relabel(train_audited, score_threshold=5)
summarize_diff(diff_table)

# Simpan diff dan dataset hasil relabel
diff_table.to_csv('../../data/result/v1/relabel_diff.csv', index=False)
train_relabeled.to_csv('../../data/result/v1/train_relabeled.csv', index=False)
print("\nDetail perubahan disimpan di data/result/v1/relabel_diff.csv")
print("Data hasil relabel disimpan di data/result/v1/train_relabeled.csv")

# Terapkan data relabeled ke train aktif
train = train_relabeled.copy()


=== Distribusi skor kecurigaan ===
suspect_score
11       1
8       24
7        1
6        1
5       34
4       10
3      129
2      134
1      132
0     7934
Name: count, dtype: int64

Skor >=5 (relabel otomatis)      : 61 baris
Skor 3-4 (verifikasi manual)     : 139 baris
Skor 1-2 (biarkan, terlalu lemah): 266 baris

Hasil audit disimpan di data/result/train_audit.csv
=== Total baris yang benar-benar berubah label: 61 ===

=== Perubahan per pasangan (label lama -> label baru) ===
category_before  category_after 
phishing         online gambling    20
malware          online gambling    14
spam             online gambling    12
other            online gambling     8
online gambling  phishing            6
spam             phishing            1
dtype: int64

Detail perubahan disimpan di data/result/relabel_diff.csv
Data hasil relabel disimpan di data/result/train_relabeled.csv


## 5. Feature Engineering: Datetime & Karakteristik Domain
Melakukan ekstraksi fitur temporal dari kolom `discovered` dan `registration_date`:
- `discovered_hour`: Jam saat entri pertama kali ditemukan (0-23).
- `discovered_dayofweek`: Hari dalam seminggu (0 = Senin, 6 = Minggu).
- `domain_age_days`: Usia domain (selisih hari antara `discovered` dan `registration_date`).
- `domain_age_is_negative`: Indikator anomali (1 jika `discovered < registration_date`, 0 jika normal; sering kali merupakan indikator kuat modus phishing).


In [8]:
# Konversi discovered & registration_date ke format datetime
train['discovered'] = pd.to_datetime(train['discovered'], format='%m/%d/%Y %H:%M')
train['registration_date'] = pd.to_datetime(train['registration_date'], format='%m/%d/%Y')

test['discovered'] = pd.to_datetime(test['discovered'], format='%m/%d/%Y %H:%M')
test['registration_date'] = pd.to_datetime(test['registration_date'], format='%m/%d/%Y')

# Fitur turunan waktu (jam dan hari)
train['discovered_hour'] = train['discovered'].dt.hour
train['discovered_dayofweek'] = train['discovered'].dt.dayofweek
test['discovered_hour'] = test['discovered'].dt.hour
test['discovered_dayofweek'] = test['discovered'].dt.dayofweek

# Selisih hari (umur domain)
train['domain_age_days'] = (train['discovered'] - train['registration_date']).dt.days
test['domain_age_days'] = (test['discovered'] - test['registration_date']).dt.days

# Flag untuk kasus discovered < registration_date (sinyal kuat ke phishing)
train['domain_age_is_negative'] = (train['domain_age_days'] < 0).astype(int)
test['domain_age_is_negative'] = (test['domain_age_days'] < 0).astype(int)

# Tampilkan preview fitur hasil rekayasa
train[['discovered', 'registration_date', 'discovered_hour', 'discovered_dayofweek', 'domain_age_days', 'domain_age_is_negative']].head()


,discovered,registration_date,discovered_hour,discovered_dayofweek,domain_age_days,domain_age_is_negative
0,2024-05-01 21:52:00,2026-05-14,21,2,-743.0,1
1,2024-08-05 20:41:00,2009-05-11,20,0,5565.0,0
2,2024-06-14 07:05:00,2008-03-13,7,4,5937.0,0
3,2024-08-04 06:20:00,2009-07-28,6,6,5486.0,0
4,2024-08-04 16:10:00,2012-01-13,16,6,4587.0,0


In [9]:
# ============================================================
# Fitur IP: ip_frequency, ip_subnet_frequency, ip_is_missing
# WAJIB: hitung referensi (freq map) HANYA dari train,
# baru diterapkan (map) ke train maupun test
# ============================================================

# --- Hitung referensi dari TRAIN saja ---
ip_freq_map = train['ip'].value_counts().to_dict()

train['ip_prefix24'] = train['ip'].str.rsplit('.', n=1).str[0]
test['ip_prefix24'] = test['ip'].str.rsplit('.', n=1).str[0]

subnet_freq_map = train['ip_prefix24'].value_counts().to_dict()

# --- Terapkan ke train ---
train['ip_frequency'] = train['ip'].map(ip_freq_map).fillna(0)
train['ip_subnet_frequency'] = train['ip_prefix24'].map(subnet_freq_map).fillna(0)
train['ip_is_missing'] = train['ip'].isna().astype(int)

# --- Terapkan ke test (pakai kamus yang SAMA dari train) ---
test['ip_frequency'] = test['ip'].map(ip_freq_map).fillna(0)
test['ip_subnet_frequency'] = test['ip_prefix24'].map(subnet_freq_map).fillna(0)
test['ip_is_missing'] = test['ip'].isna().astype(int)

# --- Bersihkan kolom bantu yang gak dipakai lagi ---
train = train.drop(columns=['ip_prefix24'])
test = test.drop(columns=['ip_prefix24'])

print(train[['ip','ip_frequency','ip_subnet_frequency','ip_is_missing']].head())

                ip  ip_frequency  ip_subnet_frequency  ip_is_missing
0              NaN           0.0                  0.0              1
1    103.162.68.84          79.0                 79.0              0
2     103.18.117.8          12.0                 51.0              0
3    103.113.3.225          87.0                 87.0              0
4  103.170.105.197          53.0                 65.0              0
